# Initiation à l'API Berserk

L'objectif de ce notebook est de comprendre comment utiliser l'API Berserk.

## 1. Imports de libraries

In [101]:
from dotenv import load_dotenv

import os
import sys
sys.path.append(os.path.abspath(".."))

from pprint import pprint

import berserk
import pandas as pd 

## 2. Chargement du token

In [102]:
load_dotenv("../.env")

token = os.getenv("LICHESS_API_TOKEN")

if not token:
    raise ValueError("LICHESS_API_TOKEN introuvable dans le fichier .env")

print("Token chargé avec succès")

Token chargé avec succès


## 3. Connexion à l'API Berserk

In [103]:
session = berserk.TokenSession(token)
client = berserk.Client(session=session)

print("Client Berserk créé avec succès")

Client Berserk créé avec succès


## 4. Affichage des informations de mon profil Lichess

In [104]:
account = client.account.get()
pprint(account)

{'blocking': False,
 'count': {'all': 1,
           'bookmark': 0,
           'draw': 0,
           'import': 0,
           'loss': 0,
           'me': 0,
           'playing': 0,
           'rated': 1,
           'win': 1},
 'createdAt': datetime.datetime(2026, 3, 17, 18, 3, 37, 330000, tzinfo=datetime.timezone.utc),
 'followable': True,
 'following': False,
 'id': 'raydennnnnnnnn',
 'perfs': {'blitz': {'games': 1,
                     'prog': 0,
                     'prov': True,
                     'rating': 1674,
                     'rd': 297},
           'bullet': {'games': 0,
                      'prog': 0,
                      'prov': True,
                      'rating': 1500,
                      'rd': 500},
           'classical': {'games': 0,
                         'prog': 0,
                         'prov': True,
                         'rating': 1500,
                         'rd': 500},
           'correspondence': {'games': 0,
                              'prog'

## 5. Importation des parties d'un joueur

Dans cet exemple, il est importé les parties du joueur Chesstam. Un maximim de 5 parties sont importées. Seules des parties jouées en mode rapide sont considérées.

In [105]:
username = "thecaptv"

games = client.games.export_by_player(
    username,
    max=150,
    perf_type="rapid",
    opening=True,
)

games = list(games)



print(f"{len(games)} parties récupérées")

150 parties récupérées


## 6. Inspection d'une partie

Cette première commande permet d'extraire toutes les informations d'une partie. L'API a converti le format PGN en JSON.

In [106]:
pprint(games[0])



{'clock': {'increment': 0, 'initial': 600, 'totalTime': 600},
 'createdAt': datetime.datetime(2026, 3, 21, 18, 23, 36, 983000, tzinfo=datetime.timezone.utc),
 'id': 'UzZTFaBc',
 'lastMoveAt': datetime.datetime(2026, 3, 21, 18, 37, 54, 138000, tzinfo=datetime.timezone.utc),
 'moves': 'd4 d5 c4 c6 Nc3 Nf6 Bg5 e6 e3 Bb4 Qc2 h6 Bxf6 Qxf6 Nf3 c5 a3 Bxc3+ '
          'Qxc3 cxd4 exd4 Nc6 cxd5 exd5 Bb5 Qe6+ Ne5 Bd7 O-O Nxe5 dxe5 Bxb5 '
          'Qb4 Bxf1 Rxf1 b6 Qb5+ Kd8 Rc1 Qxe5 Qc6 Qb8 Qxd5+ Ke8 Re1+ Kf8 f4 '
          'Qd8 Qc6 Rc8 Qb7 Rc7 Qe4 Re7 Qb4 Qe8 g4 f6 f5 Kf7 Qc4+ Kf8 Qb4 g6 '
          'fxg6 Rg8 Re6 Rxg6 h4 Rxg4+ Kh2 Rxb4',
 'opening': {'eco': 'D10', 'name': 'Slav Defense', 'ply': 5},
 'perf': 'rapid',
 'players': {'black': {'rating': 1364,
                       'ratingDiff': 6,
                       'user': {'id': 'polhovine', 'name': 'polhovine'}},
             'white': {'rating': 1366,
                       'ratingDiff': -6,
                       'user': {'id': 'thecaptv', 

Cette seconde commande permet d'extraire les différents champs d'une partie.

In [107]:
print(sorted(games[0].keys()))

for key in sorted(games[0].keys()):
    print(key)

['clock', 'createdAt', 'id', 'lastMoveAt', 'moves', 'opening', 'perf', 'players', 'rated', 'source', 'speed', 'status', 'variant', 'winner']
clock
createdAt
id
lastMoveAt
moves
opening
perf
players
rated
source
speed
status
variant
winner


Les prochaines commandes permettent d'afficher les sous-champs des champs "players" et "opening".

In [108]:
for key in sorted(games[0]["players"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"].keys()):
    print(key)

print("\n")

for key in sorted(games[0]["players"]["white"]["user"].keys()):
    print(key)

print("\n")

black
white


rating
ratingDiff
user


id
name




In [109]:
for key in sorted(games[0]["opening"].keys()):
    print(key)

eco
name
ply


# 7. Applatissement d'une partie

La commande suivante éxécute une fonction permettant de convertir una partie brute en une structure plate.

In [110]:
from src.ingestion.flatten_games import flatten_game



flat = flatten_game(games[0])
flat

{'game_id': 'UzZTFaBc',
 'datetime_utc': datetime.datetime(2026, 3, 21, 18, 23, 36, 983000, tzinfo=datetime.timezone.utc),
 'perf': 'rapid',
 'speed': 'rapid',
 'rated': True,
 'source': 'pool',
 'status': 'resign',
 'winner': 'black',
 'white_id': 'thecaptv',
 'white_name': 'thecaptv',
 'black_id': 'polhovine',
 'black_name': 'polhovine',
 'white_rating_before': 1366,
 'black_rating_before': 1364,
 'white_rating_diff': -6,
 'black_rating_diff': 6,
 'white_rating_after': 1360,
 'black_rating_after': 1370,
 'opening_eco': 'D10',
 'opening_name': 'Slav Defense',
 'clock_initial': 600,
 'clock_increment': 0,
 'has_increment': 0,
 'moves': 'd4 d5 c4 c6 Nc3 Nf6 Bg5 e6 e3 Bb4 Qc2 h6 Bxf6 Qxf6 Nf3 c5 a3 Bxc3+ Qxc3 cxd4 exd4 Nc6 cxd5 exd5 Bb5 Qe6+ Ne5 Bd7 O-O Nxe5 dxe5 Bxb5 Qb4 Bxf1 Rxf1 b6 Qb5+ Kd8 Rc1 Qxe5 Qc6 Qb8 Qxd5+ Ke8 Re1+ Kf8 f4 Qd8 Qc6 Rc8 Qb7 Rc7 Qe4 Re7 Qb4 Qe8 g4 f6 f5 Kf7 Qc4+ Kf8 Qb4 g6 fxg6 Rg8 Re6 Rxg6 h4 Rxg4+ Kh2 Rxb4',
 'ply_count': 72}

# 8. Conversion en une partie du point de vue du joueur

La commande suivante fait appel à une fonction permettant de transformer une partie aplatie en 2 rangées (1 par joueur).

In [111]:
from src.ingestion.player_view import game_to_player_rows

flat = flatten_game(games[1])
rows = game_to_player_rows(flat)

rows

[{'game_id': '3p8Qgo6D',
  'datetime_utc': datetime.datetime(2026, 3, 21, 18, 13, 26, 840000, tzinfo=datetime.timezone.utc),
  'weekday': 5,
  'player_id': 'danielfdezdelucas',
  'player_name': 'danielfdezdelucas',
  'opponent_id': 'thecaptv',
  'opponent_name': 'thecaptv',
  'color_player': 'white',
  'result_player': 0.0,
  'elo_player_before': 1311,
  'elo_player_after': 1306,
  'elo_diff_player': -5,
  'termination_type': 'resign',
  'has_increment': 0,
  'opening_family': "Petrov's Defense",
  'opening_eco': 'C42',
  'opening_name': "Petrov's Defense: Italian Variation",
  'ply_count': 58,
  'perf': 'rapid',
  'speed': 'rapid',
  'rated': True,
  'source': 'pool'},
 {'game_id': '3p8Qgo6D',
  'datetime_utc': datetime.datetime(2026, 3, 21, 18, 13, 26, 840000, tzinfo=datetime.timezone.utc),
  'weekday': 5,
  'player_id': 'thecaptv',
  'player_name': 'thecaptv',
  'opponent_id': 'danielfdezdelucas',
  'opponent_name': 'danielfdezdelucas',
  'color_player': 'black',
  'result_player': 

# 9. Table complète des parties du joueur

La commande suivante permet de convertir un ensemble de parties brutes en un ensemble de parties aplaties en 2 rangées (format de la section précédente).

In [112]:
from src.ingestion.build_player_games import build_player_games

df_player_games = build_player_games(games)

df_player_games = df_player_games[df_player_games["player_id"] == "thecaptv"].reset_index(drop=True)

print(len(games))

print(len(df_player_games))

print(df_player_games.columns)

df_player_games.head(150)



150
150
Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source'],
      dtype='str')


,game_id,datetime_utc,weekday,player_id,player_name,opponent_id,opponent_name,color_player,result_player,elo_player_before,...,termination_type,has_increment,opening_family,opening_eco,opening_name,ply_count,perf,speed,rated,source
0,UzZTFaBc,2026-03-21 18:23:36.983000+00:00,5,thecaptv,thecaptv,polhovine,polhovine,white,0.0,1366.0,...,resign,0,Slav Defense,D10,Slav Defense,72,rapid,rapid,True,pool
1,3p8Qgo6D,2026-03-21 18:13:26.840000+00:00,5,thecaptv,thecaptv,danielfdezdelucas,danielfdezdelucas,black,1.0,1361.0,...,resign,0,Petrov's Defense,C42,Petrov's Defense: Italian Variation,58,rapid,rapid,True,pool
2,vu6lRGN7,2026-03-21 18:07:57.538000+00:00,5,thecaptv,thecaptv,chesstam,Chesstam,white,0.0,1367.0,...,resign,0,Queen's Gambit Declined,D06,Queen's Gambit Declined: Marshall Defense,76,rapid,rapid,True,pool
3,SZqlUpqw,2026-03-21 18:02:32.045000+00:00,5,thecaptv,thecaptv,aydar15,Aydar15,black,0.0,1373.0,...,resign,0,Petrov's Defense,C42,Petrov's Defense,24,rapid,rapid,True,pool
4,6G65yW7j,2026-03-21 17:44:50.765000+00:00,5,thecaptv,thecaptv,emmitbrown2,emmitbrown2,white,0.0,1380.0,...,mate,0,Queen's Gambit Accepted,D20,"Queen's Gambit Accepted: Central Variation, Gr...",76,rapid,rapid,True,pool
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,2EVhw3z5,2026-02-09 23:36:06.026000+00:00,0,thecaptv,thecaptv,jygn,jygn,black,0.0,1242.0,...,resign,0,Queen's Pawn Game,D00,"Queen's Pawn Game: Chigorin Variation, Alburt ...",55,rapid,rapid,True,pool
146,j0wJuhdD,2026-02-07 17:25:15.393000+00:00,5,thecaptv,thecaptv,black_mamba5,black_mamba5,white,0.5,1243.0,...,draw,0,Slav Defense,D10,Slav Defense,129,rapid,rapid,True,pool
147,x9THHqFu,2026-02-07 17:19:12.539000+00:00,5,thecaptv,thecaptv,zero_akhi,zero_akhi,black,0.0,1291.0,...,resign,0,Ruy Lopez,C67,"Ruy Lopez: Berlin Defense, Rio Gambit Accepted",27,rapid,rapid,True,pool
148,arero8rX,2026-02-07 16:55:34.879000+00:00,5,thecaptv,thecaptv,kamablog,Kamablog,black,1.0,1262.0,...,mate,0,Vienna Game,C25,Vienna Game: Anderssen Defense,126,rapid,rapid,True,pool


# 10. Implémentation des features partie "entre-parties"

In [113]:
from src.features.temporal_features import add_basic_temporal_features

df_player_games_v2 = add_basic_temporal_features(df_player_games)

df_player_games_v2[["player_id", "index", "delay_previous_game", "streak_before", "streak_after"]].head(150)


,player_id,index,delay_previous_game,streak_before,streak_after
149,thecaptv,0,NaN,NaN,1
148,thecaptv,1,335.060,1.0,2
147,thecaptv,2,1417.660,2.0,-1
146,thecaptv,3,362.854,-1.0,0
145,thecaptv,4,195050.633,0.0,-1
...,...,...,...,...,...
4,thecaptv,145,6967.795,1.0,-1
3,thecaptv,146,1061.280,-1.0,-2
2,thecaptv,147,325.493,-2.0,-3
1,thecaptv,148,329.302,-3.0,1


# 11 Création des séries temporelles

## 11.1 Série temporelle session

L'expression ci-dessous fait appel à une fonction permettant d'associer chaque partie à une session.

In [114]:
from src.features.session_features import _add_sessions

df_player_games_v3 = _add_sessions(df_player_games_v2)

df_player_games_v3[["player_id", "datetime_utc", "delay_previous_game", "session_id"]].head(30)

,player_id,datetime_utc,delay_previous_game,session_id
149,thecaptv,2026-02-07 16:49:59.819000+00:00,NaN,1
148,thecaptv,2026-02-07 16:55:34.879000+00:00,335.060,1
147,thecaptv,2026-02-07 17:19:12.539000+00:00,1417.660,1
146,thecaptv,2026-02-07 17:25:15.393000+00:00,362.854,1
145,thecaptv,2026-02-09 23:36:06.026000+00:00,195050.633,2
144,thecaptv,2026-02-10 17:51:55.457000+00:00,65749.431,3
143,thecaptv,2026-02-10 18:21:24.135000+00:00,1768.678,3
142,thecaptv,2026-02-10 18:32:37.168000+00:00,673.033,3
141,thecaptv,2026-02-10 18:36:59.403000+00:00,262.235,3
140,thecaptv,2026-02-10 18:44:53.139000+00:00,473.736,3


L'expression ci-dessous réplique le bloc précédent et intègre des features associées aux sessions.

In [115]:
from src.features.session_features import add_sessions

df_player_games_v4 = add_sessions(df_player_games_v2)

df_player_games_v4[["player_id", "session_id", "session_position", "session_size", "session_score", "datetime_utc", "session_delay", "session_discrete_delay"]].head(30)

,player_id,session_id,session_position,session_size,session_score,datetime_utc,session_delay,session_discrete_delay
149,thecaptv,1,0,4,0.625000,2026-02-07 16:49:59.819000+00:00,NaN,NaN
148,thecaptv,1,1,4,0.625000,2026-02-07 16:55:34.879000+00:00,NaN,NaN
147,thecaptv,1,2,4,0.625000,2026-02-07 17:19:12.539000+00:00,NaN,NaN
146,thecaptv,1,3,4,0.625000,2026-02-07 17:25:15.393000+00:00,NaN,NaN
145,thecaptv,2,0,1,0.000000,2026-02-09 23:36:06.026000+00:00,195050.633,C
144,thecaptv,3,0,6,0.666667,2026-02-10 17:51:55.457000+00:00,65749.431,B
143,thecaptv,3,1,6,0.666667,2026-02-10 18:21:24.135000+00:00,65749.431,B
142,thecaptv,3,2,6,0.666667,2026-02-10 18:32:37.168000+00:00,65749.431,B
141,thecaptv,3,3,6,0.666667,2026-02-10 18:36:59.403000+00:00,65749.431,B
140,thecaptv,3,4,6,0.666667,2026-02-10 18:44:53.139000+00:00,65749.431,B


## 11.2 Série temporelle games_per_day

Même principe que pour la série sessions sauf qu'ici une session de jeu est une journée.

In [116]:
from src.features.day_features import add_day_features

df_player_games_v5 = add_day_features(df_player_games_v4)

df_player_games_v5[["player_id", "datetime_utc", "day_date_utc", "day_n_games", "day_score", "day_position"]].head(30)

,player_id,datetime_utc,day_date_utc,day_n_games,day_score,day_position
149,thecaptv,2026-02-07 16:49:59.819000+00:00,2026-02-07,4,0.625000,0
148,thecaptv,2026-02-07 16:55:34.879000+00:00,2026-02-07,4,0.625000,1
147,thecaptv,2026-02-07 17:19:12.539000+00:00,2026-02-07,4,0.625000,2
146,thecaptv,2026-02-07 17:25:15.393000+00:00,2026-02-07,4,0.625000,3
145,thecaptv,2026-02-09 23:36:06.026000+00:00,2026-02-09,1,0.000000,0
144,thecaptv,2026-02-10 17:51:55.457000+00:00,2026-02-10,11,0.454545,0
143,thecaptv,2026-02-10 18:21:24.135000+00:00,2026-02-10,11,0.454545,1
142,thecaptv,2026-02-10 18:32:37.168000+00:00,2026-02-10,11,0.454545,2
141,thecaptv,2026-02-10 18:36:59.403000+00:00,2026-02-10,11,0.454545,3
140,thecaptv,2026-02-10 18:44:53.139000+00:00,2026-02-10,11,0.454545,4


## 11.3 Série temporelle games_per_week

Même principe que pour la série sessions sauf qu'ici une session de jeu est une semaine.

In [117]:
from src.features.week_features import add_week_features

df_player_games_v6 = add_week_features(df_player_games_v5)

df_player_games_v6[["player_id", "datetime_utc", "week_id", "week_n_games", "week_score", "week_position", "termination_type", "result_player"]].head(30)

,player_id,datetime_utc,week_id,week_n_games,week_score,week_position,termination_type,result_player
149,thecaptv,2026-02-07 16:49:59.819000+00:00,2026-W06,4,0.625000,0,mate,1.0
148,thecaptv,2026-02-07 16:55:34.879000+00:00,2026-W06,4,0.625000,1,mate,1.0
147,thecaptv,2026-02-07 17:19:12.539000+00:00,2026-W06,4,0.625000,2,resign,0.0
146,thecaptv,2026-02-07 17:25:15.393000+00:00,2026-W06,4,0.625000,3,draw,0.5
145,thecaptv,2026-02-09 23:36:06.026000+00:00,2026-W07,34,0.411765,0,resign,0.0
144,thecaptv,2026-02-10 17:51:55.457000+00:00,2026-W07,34,0.411765,1,mate,1.0
143,thecaptv,2026-02-10 18:21:24.135000+00:00,2026-W07,34,0.411765,2,mate,0.0
142,thecaptv,2026-02-10 18:32:37.168000+00:00,2026-W07,34,0.411765,3,resign,1.0
141,thecaptv,2026-02-10 18:36:59.403000+00:00,2026-W07,34,0.411765,4,resign,0.0
140,thecaptv,2026-02-10 18:44:53.139000+00:00,2026-W07,34,0.411765,5,resign,1.0


In [118]:
print(df_player_games_v6.columns)

Index(['game_id', 'datetime_utc', 'weekday', 'player_id', 'player_name',
       'opponent_id', 'opponent_name', 'color_player', 'result_player',
       'elo_player_before', 'elo_player_after', 'elo_diff_player',
       'termination_type', 'has_increment', 'opening_family', 'opening_eco',
       'opening_name', 'ply_count', 'perf', 'speed', 'rated', 'source',
       'index', 'delay_previous_game', 'streak_after', 'streak_before',
       'new_session', 'session_id', 'session_position', 'session_size',
       'session_score', 'is_session_start', 'session_delay',
       'session_discrete_delay', 'day_date_utc', 'day_n_games', 'day_score',
       'day_position', 'week_id', 'week_n_games', 'week_score',
       'week_position'],
      dtype='str')


# 12. Features joueur

## 12.1 Features liées au style de jeu

In [119]:
from src.features.player_features import _init_player_features
from src.features.player_features import _add_style_features

features = _init_player_features(df_player_games_v6) 
features_v1 = _add_style_features(df_player_games_v6, features)

features_v1[[
    "mean_ply_count",
    "main_opening_white",
    "main_opening_black",
    "opening_diversity",
    "opening_concentration"
]].head()

,mean_ply_count,main_opening_white,main_opening_black,opening_diversity,opening_concentration
player_id,,,,,
thecaptv,64.053333,Queen's Gambit Declined,Benoni Defense,2.399391,0.226358


## 12.2 Features liées au comportement en fin de partie

In [120]:
from src.features.player_features import _add_endgame_behavior_features

features_v2 = _add_endgame_behavior_features(df_player_games_v6, features_v1)

features_v2[[
    "draw_ratio",
    "abandon_rate",
    "time_loss_rate"
]].head()

,draw_ratio,abandon_rate,time_loss_rate
player_id,,,
thecaptv,0.026667,0.253333,0.0


## 12.3 Features liées aux streaks

In [121]:
from src.features.player_features import _add_streak_features

features_v3 = _add_streak_features(df_player_games_v6, features_v2)

features_v2[[
    "score_when_winstreak",
    "score_when_losestreak",
    "streak_resilience_bias",
    "abandon_rate_when_losestreak",
    "time_loss_rate_when_losestreak",
    "delay_ratio_when_winstreak",
    "delay_ratio_when_losestreak"
]].head()

,score_when_winstreak,score_when_losestreak,streak_resilience_bias,abandon_rate_when_losestreak,time_loss_rate_when_losestreak,delay_ratio_when_winstreak,delay_ratio_when_losestreak
player_id,,,,,,,
thecaptv,0.62,0.541667,0.078333,0.333333,0.0,1.424383,0.325237


## 12.4 Features liées au rythme de jeu global

In [122]:
from src.features.player_features import _add_global_rhythm_features

features_v4 = _add_global_rhythm_features(df_player_games_v6, features_v3)

features_v4[[
    "cv_games_interval",
    "cv_games_per_day",
    "cv_games_per_week"
]].head()

,cv_games_interval,cv_games_per_day,cv_games_per_week
player_id,,,
thecaptv,2.441111,0.849141,0.614311


## 12.5 Features liées à la structure des sessions

In [123]:
from src.features.player_features import _add_session_structure_features

features_v5 = _add_session_structure_features(df_player_games_v6, features_v4)

features_v5[[
    "cv_sessions_interval",
    "entropy_sessions_interval",
    "mean_games_per_session",
    "cv_games_per_session"
]].head()

,cv_sessions_interval,entropy_sessions_interval,mean_games_per_session,cv_games_per_session
player_id,,,,
thecaptv,1.349518,1.119608,2.542373,0.87136


## 12.6 Features liées au contexte de jeu

In [ ]:
from src.features.player_features import _add_context_features

features_v6 = _add_context_features(df_player_games_v6, features_v5)

features_v6[[
    "color_bias",
    "increment_game_ratio",
    "weekday_bias"
]].head()

has_increment
0    150
Name: count, dtype: int64